# 01 — Data Understanding & Schema Profiling
**Project:** Axlero Solutions — Credit Risk Scoring and Loan Default Analysis  
**Module:** Data Understanding (Foundation for Member 1 — Darshan Patil)

## 1. Objectives
1. Load the immutable raw dataset (`data/raw/credit_risk_dataset.csv`, 32,581 records × 12 columns) using portable project-relative paths.
2. Profile variable data types, domain definitions, summary statistics, and target distribution (`loan_status`).
3. Audit exact duplicate records (`keep='first'` vs `keep=False`), missing value patterns (`person_emp_length` and `loan_int_rate`), and impossible domain anomalies (`person_age > 100`, `person_emp_length > 60`).

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.load_data import load_raw_data, summarize_dataset

df_raw = load_raw_data()
print(f"Raw Dataset Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
df_raw.head(10)

Raw Dataset Shape: 32,581 rows x 12 columns


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4
5,21,9900,OWN,2.0,VENTURE,A,2500,7.14,1,0.25,N,2
6,26,77100,RENT,8.0,EDUCATION,B,35000,12.42,1,0.45,N,3
7,24,78956,RENT,5.0,MEDICAL,B,35000,11.11,1,0.44,N,4
8,24,83000,RENT,8.0,PERSONAL,A,35000,8.90,1,0.42,N,2
9,21,10000,OWN,6.0,VENTURE,D,1600,14.74,1,0.16,N,3


## 2. Schema & Variable Dictionary
| Column | Type | Description |
|---|---|---|
| `person_age` | Integer | Applicant age in years |
| `person_income` | Integer | Annual gross/net income |
| `person_home_ownership` | Categorical | Housing tenure (`RENT`, `MORTGAGE`, `OWN`, `OTHER`) |
| `person_emp_length` | Float | Employment length in years |
| `loan_intent` | Categorical | Stated loan purpose (`EDUCATION`, `MEDICAL`, `VENTURE`, `PERSONAL`, `HOMEIMPROVEMENT`, `DEBTCONSOLIDATION`) |
| `loan_grade` | Categorical | Underwriting risk grade (`A` = lowest risk to `G` = highest risk) |
| `loan_amnt` | Integer | Requested loan principal amount |
| `loan_int_rate` | Float | Annual interest rate (%) |
| `loan_status` | Binary Target | `1` = Default, `0` = Non-Default / Repaid |
| `loan_percent_income` | Float | Stated ratio of loan amount to annual income |
| `cb_person_default_on_file` | Categorical | Historical credit bureau default indicator (`Y`, `N`) |
| `cb_person_cred_hist_length` | Integer | Length of credit history in years |

In [2]:
summary = summarize_dataset(df_raw)
print("Target Distribution (Raw Dataset):")
print(pd.Series(summary["target_distribution"]))

missing_df = pd.DataFrame({
    "Missing Count": df_raw.isnull().sum(),
    "Missing %": (df_raw.isnull().mean() * 100).round(3),
    "Dtype": df_raw.dtypes.astype(str)
})
display(missing_df)

Target Distribution (Raw Dataset):
non_default_0       25473.0000
default_1            7108.0000
default_rate_pct       21.8164
dtype: float64


,Missing Count,Missing %,Dtype
person_age,0,0.000,int64
person_income,0,0.000,int64
person_home_ownership,0,0.000,str
person_emp_length,895,2.747,float64
loan_intent,0,0.000,str
loan_grade,0,0.000,str
loan_amnt,0,0.000,int64
loan_int_rate,3116,9.564,float64
loan_status,0,0.000,int64
loan_percent_income,0,0.000,float64


## 3. Resolving Duplicate Record Counts & Initial Anomaly Audit
In initial notebook outputs, two duplicate figures appeared:
- `df.duplicated(keep='first').sum() = 165` (redundant duplicate copies to be dropped).
- `df.duplicated(keep=False).sum() = 330` (all rows belonging to the 165 duplicated pairs: 165 originals + 165 copies).

In [3]:
print("Redundant duplicate copies (keep='first'):", df_raw.duplicated(keep="first").sum())
print("All rows involved in duplicate pairs (keep=False):", df_raw.duplicated(keep=False).sum())

print("\nNumeric Summary Statistics (Raw):")
display(df_raw.describe().T.round(2))

print("\nImpossible Age Records (person_age > 100):")
display(df_raw[df_raw["person_age"] > 100])

print("\nImpossible Employment Length Records (person_emp_length > 60):")
display(df_raw[df_raw["person_emp_length"] > 60])

Redundant duplicate copies (keep='first'): 165
All rows involved in duplicate pairs (keep=False): 330

Numeric Summary Statistics (Raw):


,count,mean,std,min,25%,50%,75%,max
person_age,32581.0,27.73,6.35,20.00,23.00,26.00,30.00,144.00
person_income,32581.0,66074.85,61983.12,4000.00,38500.00,55000.00,79200.00,6000000.00
person_emp_length,31686.0,4.79,4.14,0.00,2.00,4.00,7.00,123.00
loan_amnt,32581.0,9589.37,6322.09,500.00,5000.00,8000.00,12200.00,35000.00
loan_int_rate,29465.0,11.01,3.24,5.42,7.90,10.99,13.47,23.22
loan_status,32581.0,0.22,0.41,0.00,0.00,0.00,0.00,1.00
loan_percent_income,32581.0,0.17,0.11,0.00,0.09,0.15,0.23,0.83
cb_person_cred_hist_length,32581.0,5.80,4.06,2.00,3.00,4.00,8.00,30.00



Impossible Age Records (person_age > 100):


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
81,144,250000,RENT,4.0,VENTURE,C,4800,13.57,0,0.02,N,3
183,144,200000,MORTGAGE,4.0,EDUCATION,B,6000,11.86,0,0.03,N,2
575,123,80004,RENT,2.0,EDUCATION,B,20400,10.25,0,0.25,N,3
747,123,78000,RENT,7.0,VENTURE,B,20000,NaN,0,0.26,N,4
32297,144,6000000,MORTGAGE,12.0,PERSONAL,C,5000,12.73,0,0.00,N,25



Impossible Employment Length Records (person_emp_length > 60):


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
210,21,192000,MORTGAGE,123.0,VENTURE,A,20000,6.54,0,0.10,N,4
